In [3]:
from dotenv import load_dotenv
import os
from openai import OpenAI

load_dotenv()

# Connect to Groq (same as Module 1)
client = OpenAI(
api_key=os.environ.get("GROQ_API_KEY"),
base_url="https://api.groq.com/openai/v1"
)
print("Setup complete!")

Setup complete!


In [4]:
from gitsource import GithubRepositoryDataReader
reader = GithubRepositoryDataReader(
repo_owner="DataTalksClub",
repo_name="llm-zoomcamp",
commit_id="8c1834d",
allowed_extensions={"md"},
filename_filter=lambda path: "/lessons/" in path,
)
files = reader.read()
# Parse files into documents
documents = []
for file in files:
    doc = file.parse()
    documents.append(doc)

print(f"Total documents loaded: {len(documents)}")
print(documents[0])

Total documents loaded: 72
{'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuati

In [7]:
from minsearch import Index
# Build index — content is text field, filename is keyword field
index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

index.fit(documents)
print("Index ready!")

# Search with the exact query from the homework
query = "How does the agentic loop keep calling the model until it stops?"

search_results = index.search(
    query,
    boost_dict={"content": 1.0},
    num_results=5
)

# Print filename of FIRST result — this is your Q2 answer!
print("Q2 Answer:")
print(search_results[0]['filename'])


Index ready!
Q2 Answer:
01-agentic-rag/lessons/14-agentic-loop.md


In [10]:
INSTRUCTIONS = """
You are a course teaching assistant.
Answer the student question based on the context.
If not in context, say: I don't know.
"""

def build_context(search_results):
    lines = []
    for doc in search_results:
        lines.append("File: " + doc['filename'])
        lines.append("Content: " + doc['content'])
        lines.append("")
    return "\n".join(lines).strip()

def build_prompt(question, search_results):
    context = build_context(search_results)
    prompt = "Question: " + question + "\n\nContext: " + context
    return prompt.strip()

def llm(prompt):
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[
            {"role": "system", "content": INSTRUCTIONS},
            {"role": "user", "content": prompt}
        ]
    )
    return response

query = "How does the agentic loop keep calling the model until it stops?"
results = index.search(query, num_results=5)
prompt = build_prompt(query, results)
response = llm(prompt)

print("Q3 Answer:")
print(response.usage.prompt_tokens)


Q3 Answer:
7211


In [11]:
from gitsource import chunk_documents

# Chunk all documents
# size=2000 means each chunk is 2000 characters
# step=1000 means window slides 1000 chars forward each time
# So consecutive chunks overlap by 1000 characters
chunks = chunk_documents(
    documents, 
    size=2000,
    step=1000
)

# Q4 Answer — how many chunks?
print("Q4 Answer:")
print(len(chunks))
# Look at one chunk to understand structure
print(chunks[0])

Q4 Answer:
295
{'start': 0, 'content': '# Introduction\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=rQYyFxf1FWw&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nIn this module, we\'ll build a working Retrieval-Augmented\nGeneration (RAG) system from scratch, step by step.\n\nWe write everything in plain Python. We build a small search index by\nhand and call the LLM ourselves. I want you to see every piece first.\nThat way you know what a framework does for you before you reach for\none.\n\nPlaces where you can find me:\n\n- [My substack](https://alexeyondata.substack.com/)\n- [LinkedIn](https://www.linkedin.com/in/agrigorev/)\n- [X](https://x.com/Al_Grigor)\n\n## LLMs\n\nAn LLM (Large Language Model) is a neural network trained on massive\namounts of text. Given a prompt, it generates a continuation - a\nplausible next piece of text.\n\nThink of your phone. When you type "how are" in WhatsApp, it suggests\n"you" as the next word. "How are you" is the most common continuati

In [12]:
from minsearch import Index

# Build NEW index from CHUNKS
chunk_index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

chunk_index.fit(chunks)
print("Chunk index ready!")

# Run same RAG query but with chunk index
query = "How does the agentic loop keep calling the model until it stops?"

chunk_results = chunk_index.search(query, num_results=5)
chunk_prompt = build_prompt(query, chunk_results)
chunk_response = llm(chunk_prompt)

chunk_tokens = chunk_response.usage.prompt_tokens

print("Q3 tokens (original):", 7211)
print("Q5 tokens (chunked):", chunk_tokens)
print("Ratio:", round(7211 / chunk_tokens, 1))


Chunk index ready!
Q3 tokens (original): 7211
Q5 tokens (chunked): 2339
Ratio: 3.1


In [17]:
search_count = 0

def search_tool(query):
    global search_count
    search_count += 1
    results = chunk_index.search(query, num_results=3)
    return build_context(results)

tools = [{
    "type": "function",
    "function": {
        "name": "search",
        "description": "Search course lessons for information. Make multiple searches.",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {"type": "string"}
            },
            "required": ["query"]
        }
    }
}]

import json

messages = [
    {"role": "system", "content": "You are a course teaching assistant. Answer using the search tool. Make multiple searches with different keywords before answering."},
    {"role": "user", "content": "How does the agentic loop work and how is it different from plain RAG?"}
]

search_count = 0

for i in range(10):
    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=messages,
        tools=tools
    )
    msg = response.choices[0].message
    if msg.tool_calls:
        messages.append(msg)
        for tc in msg.tool_calls:
            result = search_tool(tc.function.arguments)
            messages.append({
                "role": "tool",
                "tool_call_id": tc.id,
                "content": result
            })
    else:
        print("Agent answer:", msg.content)
        break

print("")
print("Q6 Answer - Search called", search_count, "times")


Agent answer: The agentic loop works by having an agent that decides what to do, in what order, based on the goal. It is different from plain RAG in that it uses a loop to continually ask for and receive information until a final answer is reached, whereas RAG is a pipeline with three steps: search, build prompt, and LLM call. The agentic loop is more flexible and can handle dynamic changing information, whereas RAG is more suitable for fixed sequences of steps.

Q6 Answer - Search called 4 times
